# 16 - BioPWM: the protein writes its own motif detector

**What.** Instead of *fusing* the protein into a fixed head, a hypernetwork makes the protein **generate the RNA-side motif detector**, and constrains that detector to a **proper biophysical PWM** - so the motif is the model's explicit, readable latent. The PWM scans the **raw RNA** (no PARNET), making the protein the *only* path to the prediction: leakage-free by construction.

**Why.** Two project problems share one solution. (1) Interpretability: post-hoc attention collapses zero-shot (notebook 15: 0.88x/0.55x, below control), so a model whose motif IS a parameter you read is the only interpretability that survives shift. (2) Leakage: the RNA-only baseline is ~47% frozen-PARNET memorization; a protein->PWM->RNA-sequence predictor bypasses PARNET entirely.

**Data.** Lab pureclip binding panel (K=68 eCLIP tracks / 45 RBPs), per-residue ESM (`perres32`), real lab PARNET `parnet.7m-0.0`. 5 seeds. Every number is a protein-vs-shuffle GAP with a CROSS-FAMILY (derangement) and a harder WITHIN-FAMILY shuffle; paired bootstrap CI + sign test across RBPs. The specificity track is leakage-free by construction (scans RNA one-hot, no PARNET).

## Definitions (the maths)

**Recognition-code generator (per-residue, RBD-weighted).** Protein per-residue tokens $E\in\mathbb{R}^{R\times d}$; $K$ learned position-queries $Q\in\mathbb{R}^{K\times d}$ attend over residues (RBD residues win): $A=\mathrm{softmax}_r(QE^\top/\sqrt d)$, context $C=A E\in\mathbb{R}^{K\times d}$, then a SHARED amino-acid->base code $T\in\mathbb{R}^{d\times4}$ gives log-odds $W=CT\in\mathbb{R}^{K\times4}$.

**Biophysics (Ch201/Stormo/Foat).** PWM $P=\mathrm{softmax}_{\text{base}}(W)$; log-odds $W=\log P-\log b$ ($b=\tfrac14$); occupancy on RNA one-hot $X$: $\mathrm{aff}=\tfrac1\beta\log\sum_x e^{\beta\,(W\star X)(x)}$ (Foat soft-max of binding energies; **logsumexp, never a saturating max** - Ch206's bug). Binding logit $=\alpha\,\mathrm{aff}+b_0$ (leakage-free), optionally $+\,h(\text{PARNET pooled})$ (combined; the protein-agnostic bias **cancels in the real-minus-shuffle gap**, so the gap stays leakage-free while absolute prediction uses the real PARNET rep).

**Read-out.** real protein vs CROSS-FAMILY shuffle (does the right protein write a better motif than a wrong-family one?) and vs WITHIN-FAMILY shuffle (RBP-idiosyncratic specificity); leave-out-RBP = zero-shot.

## 1 - Leakage-free BioPWM beats both shuffles, in-distribution and zero-shot

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.apply_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n
    return json.loads(p.read_text()) if p.exists() else None
def Jf(*names):
    for n in names:
        d=J(n)
        if d is not None: return d,n
    return None,None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
def rows3(d):
    R=d['rows']; return (np.array([r['real'] for r in R]),np.array([r['shuffle'] for r in R]),np.array([r['fam'] for r in R]))
di,ni=Jf('binding_biopwm_recog_indist.json','binding_biopwm_indist.json')
dr,nr=Jf('binding_biopwm_recog_rbp.json','binding_biopwm_rbp.json')
for tag,d in [('in-distribution',di),('leave-out-RBP (zero-shot)',dr)]:
    if d: print(f"{tag:26} real {d['real']:.4f} | vs cross-family {d['gap']:+.4f} ({d['n_beat']}/{d['eval_n']}, p={d['wilcoxon_p']:.1e}) | vs within-family {d['gap_fam']:+.4f} (p={d['wilcoxon_p_fam']:.1e})")

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.apply_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n
    return json.loads(p.read_text()) if p.exists() else None
def Jf(*names):
    for n in names:
        d=J(n)
        if d is not None: return d,n
    return None,None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
def rows3(d):
    R=d['rows']; return (np.array([r['real'] for r in R]),np.array([r['shuffle'] for r in R]),np.array([r['fam'] for r in R]))
di,_=Jf('binding_biopwm_recog_indist.json','binding_biopwm_indist.json')
dr,_=Jf('binding_biopwm_recog_rbp.json','binding_biopwm_rbp.json')
fig,axes=plt.subplots(1,2,figsize=(9,3.8),sharey=True)
for ax,(t,d) in zip(axes,[('in-distribution',di),('leave-out-RBP zero-shot',dr)]):
    if not d: ax.set_visible(False); continue
    r,s,f=rows3(d)
    ps.gap_violin(ax,{'protein':r,'cross-family\nshuffle':s,'within-family\nshuffle':f},ylabel='per-RBP binding auPRC',title=t,paired=True)
ps.panel_label(axes[0],'a'); ps.panel_label(axes[1],'b')
show(fig,'nb16_gaps.png','Leakage-free BioPWM: the protein writes a functional, RBP-specific motif detector')

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.apply_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n
    return json.loads(p.read_text()) if p.exists() else None
def Jf(*names):
    for n in names:
        d=J(n)
        if d is not None: return d,n
    return None,None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
def rows3(d):
    R=d['rows']; return (np.array([r['real'] for r in R]),np.array([r['shuffle'] for r in R]),np.array([r['fam'] for r in R]))
di,_=Jf('binding_biopwm_recog_indist.json','binding_biopwm_indist.json'); dr,_=Jf('binding_biopwm_recog_rbp.json','binding_biopwm_rbp.json')
display(Markdown(f'''**Result.** Leakage-free (no PARNET), the protein-generated PWM detects binding far better with the RIGHT protein than a wrong one. In-distribution: vs cross-family {di['gap']:+.4f} (p={di['wilcoxon_p']:.0e}), and crucially vs the harder WITHIN-FAMILY shuffle {di['gap_fam']:+.4f} (p={di['wilcoxon_p_fam']:.0e}) - so the PWM is RBP-IDIOSYNCRATIC, not merely family-level (where the M2 cross-attn head was borderline). Leave-out-RBP zero-shot: vs cross-family {dr['gap']:+.4f} (p={dr['wilcoxon_p']:.0e}), vs within-family {dr['gap_fam']:+.4f} (p={dr['wilcoxon_p_fam']:.0e}). Absolute auPRC ({di['real']:.3f}) sits at/below the PARNET-conditioned head ({0.122:.3f}); BioPWM is not a better predictor - its value is being LEAKAGE-FREE and INTERPRETABLE.'''))

## 2 - The combined variant uses the real PARNET rep without re-introducing the leak

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.apply_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n
    return json.loads(p.read_text()) if p.exists() else None
def Jf(*names):
    for n in names:
        d=J(n)
        if d is not None: return d,n
    return None,None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
def rows3(d):
    R=d['rows']; return (np.array([r['real'] for r in R]),np.array([r['shuffle'] for r in R]),np.array([r['fam'] for r in R]))
rows=[]
for gen,bias,lab in [('recog',0,'leakage-free'),('recog',1,'+PARNET bias')]:
    for split in ['indist','rbp']:
        nm=f"binding_biopwm_{gen}{'_bias' if bias else ''}_{split}.json"; d=J(nm)
        if d: rows.append((f"{lab}\n{split}",d['gap'],d['gap_fam'],d['real']))
if rows:
    fig,ax=plt.subplots(figsize=(6,3.6)); x=np.arange(len(rows))
    ax.bar(x-0.2,[r[1] for r in rows],0.38,color=ps.PALETTE['protein'],label='vs cross-family',edgecolor='white')
    ax.bar(x+0.2,[r[2] for r in rows],0.38,color=ps.PALETTE['family'],label='vs within-family',edgecolor='white')
    ax.axhline(0,color='#000',lw=0.8); ax.set_xticks(x); ax.set_xticklabels([r[0] for r in rows],fontsize=7)
    ax.set_ylabel('protein-vs-shuffle auPRC gap'); ax.legend(frameon=True,fontsize=7.5); ps.despine(ax)
    show(fig,'nb16_variants.png','BioPWM variants: leakage-free vs combined-with-PARNET (gap stays leakage-free)')
else: display(Markdown('_combined-bias run pending_'))

## 3 - The generated PWM IS the interpretable latent (read it, compare to ATtRACT)

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.apply_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n
    return json.loads(p.read_text()) if p.exists() else None
def Jf(*names):
    for n in names:
        d=J(n)
        if d is not None: return d,n
    return None,None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
def rows3(d):
    R=d['rows']; return (np.array([r['real'] for r in R]),np.array([r['shuffle'] for r in R]),np.array([r['fam'] for r in R]))
import numpy as np
pw=None
for nm in ['biopwm_recog_indist_pwms.npz','biopwm_recog_bias_indist_pwms.npz','biopwm_pwms_indist.npz']:
    p=OUT/nm
    if p.exists(): pw=np.load(p,allow_pickle=True); break
if pw is not None:
    P=pw['pwm']; syms=list(pw['syms'])
    # P shape (K_rbp, KW, 4) for recog, or (K_rbp, F, KW, 4) for pooled-multi; collapse to (KW,4) for an example
    ex=P[0]
    if ex.ndim==3: ex=ex[np.argmax(ex.reshape(ex.shape[0],-1).var(1))]
    ex=np.asarray(ex,float); ex=ex/ex.sum(1,keepdims=True).clip(1e-9)
    fig,ax=plt.subplots(figsize=(5,2.6))
    im=ax.imshow(ex.T,aspect='auto',cmap='viridis'); ax.set_yticks(range(4)); ax.set_yticklabels(list('ACGU'))
    ax.set_xlabel('motif position'); ax.set_title(f'Generated PWM for {syms[0]} (read directly - no attribution)')
    plt.colorbar(im,ax=ax,fraction=0.05,pad=0.02,label='P(base)')
    show(fig,'nb16_pwm.png')
    print('PWM tensor shape', P.shape)
else: display(Markdown('_PWMs pending_'))

## 4 - External fine-affinity validation (RBNS / RNAcompete) - the protein deviation is real off-eCLIP

eCLIP binding is at its within-family assay ceiling (~0.018 even for the oracle motif), so the protein refinement is read on FINE in-vitro affinity assays (isaac `rbns_validate.py` / `rnacompete_validate.py`, recognition-code protein->PWM):

- **RBNS cross-platform** (ATtRACT-trained protein->PWM predicts the RBNS-measured deviation): **+0.31 vs null +0.15, perm-p=0.011** (n=12); direct PWM beats family-mean in **7/12**.
- **RNAcompete affinity** (n=7): protein-predicted PWM Spearman **+0.167 vs family-mean -0.022**, beats family-mean **5/7**; PTBP1 (+0.67) and SRSF1 (+0.59) beat even the oracle motif.

So the protein->PWM recognition code is identifiable at affinity resolution where coarse eCLIP binding hides it.

## 5 - Multimodal BioPWM (BioPWM + PARNET fusion) on the M2 profile - head to head

The principled multimodal fusion (your question): PARNET supplies the bias/control track (learned RNA rep), the protein-BioPWM supplies the interpretable target shape, combined in the RBPNet additive multinomial mixture; the protein gap stays leakage-free (bias cancels). Run head-to-head vs FiLM and per-residue cross-attn on the SAME M2 leave-out-RBP zero-shot task, both cells.

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.apply_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n
    return json.loads(p.read_text()) if p.exists() else None
def Jf(*names):
    for n in names:
        d=J(n)
        if d is not None: return d,n
    return None,None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
def rows3(d):
    R=d['rows']; return (np.array([r['real'] for r in R]),np.array([r['shuffle'] for r in R]),np.array([r['fam'] for r in R]))
for c in ('HepG2','K562'):
    d=J(f'm2_profile_combo_{c}.json')
    if d:
        for a,v in d['archs'].items(): print(f"  {c:5} {a:7} profile-Pearson real={v['real']:.3f} | gap vs shuffle {v['gap_der']:+.3f} | vs within-family {v['gap_fam']:+.3f}")

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.apply_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n
    return json.loads(p.read_text()) if p.exists() else None
def Jf(*names):
    for n in names:
        d=J(n)
        if d is not None: return d,n
    return None,None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
def rows3(d):
    R=d['rows']; return (np.array([r['real'] for r in R]),np.array([r['shuffle'] for r in R]),np.array([r['fam'] for r in R]))
cells=[c for c in ('HepG2','K562') if J(f'm2_profile_combo_{c}.json')]
heads=['film','perres','biopwm']; lab={'film':'FiLM','perres':'per-residue\nX-attn','biopwm':'BioPWM\nmultimodal'}
fig,axes=plt.subplots(1,len(cells),figsize=(4.6*len(cells),3.6),sharey=True)
axes=np.atleast_1d(axes)
for ax,c in zip(axes,cells):
    d=J(f'm2_profile_combo_{c}.json'); x=np.arange(len(heads))
    ax.bar(x-0.2,[d['archs'][h]['gap_der'] for h in heads],0.38,color=ps.PALETTE['protein'],label='vs shuffle',edgecolor='white')
    ax.bar(x+0.2,[d['archs'][h]['gap_fam'] for h in heads],0.38,color=ps.PALETTE['family'],label='vs within-family',edgecolor='white')
    ax.axhline(0,color='#000',lw=0.8); ax.set_xticks(x); ax.set_xticklabels([lab[h] for h in heads],fontsize=7.5)
    ax.set_title(c); ps.despine(ax)
    if ax is axes[0]: ax.set_ylabel('M2 profile-Pearson gap'); ax.legend(frameon=True,fontsize=7)
show(fig,'nb16_m2_headtohead.png','M2 nt-resolution profile (zero-shot): per-residue wins, BioPWM is NULL on profile shape')

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.apply_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n
    return json.loads(p.read_text()) if p.exists() else None
def Jf(*names):
    for n in names:
        d=J(n)
        if d is not None: return d,n
    return None,None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
def rows3(d):
    R=d['rows']; return (np.array([r['real'] for r in R]),np.array([r['shuffle'] for r in R]),np.array([r['fam'] for r in R]))
h=J('m2_profile_combo_HepG2.json')
display(Markdown(f'''**Result (multimodal BioPWM on the profile - honest NULL).** On the nt-resolution profile, the BioPWM target track is **null** (HepG2 gap {h['archs']['biopwm']['gap_der']:+.3f}, real Pearson {h['archs']['biopwm']['real']:.3f} vs per-residue {h['archs']['perres']['real']:.3f}). A PWM predicts *where the motif is* (sharp occupancy) - which is a strong **binary**/affinity signal (sections 1-4) - but not the broad eCLIP **coverage shape**, so softmax(PWM-occupancy) does not match the profile. **Per-residue cross-attn remains the profile winner** (gap {h['archs']['perres']['gap_der']:+.3f}, within-family {h['archs']['perres']['gap_fam']:+.3f}). The additive bias-mixture IS the right multimodal fusion (uses PARNET + protein, leakage-free gap, interpretable), but BioPWM's inductive bias fits binary occupancy, not profile shape - different tasks, different best heads.'''))

## 6 - The generated PWMs are readable motifs (regulatory-genomics logos)

The point of BioPWM: the motif is the model's explicit latent. Here are the generated PWMs (recognition-code generator) as standard information-content sequence logos - read directly, no attribution, comparable to ATtRACT/RBPmap.

In [ ]:
import os, sys, json, pathlib
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display, Image as _Img
_here=pathlib.Path.cwd().resolve()
REPO=next((c for c in (_here,*_here.parents) if (c/'src'/'mmpartnet').is_dir()),_here)
sys.path.insert(0,str(REPO/'scripts')); import plot_style as ps; ps.apply_style()
OUT=REPO/'mmpartnet_out'; FIGD=REPO/'notebooks'/'demo'/'executed'
def J(n):
    p=OUT/n
    return json.loads(p.read_text()) if p.exists() else None
def Jf(*names):
    for n in names:
        d=J(n)
        if d is not None: return d,n
    return None,None
def show(fig,name,sup=None):
    if sup: fig.suptitle(sup,fontsize=11,fontweight='bold',y=1.02)
    fig.savefig(str(FIGD/name),bbox_inches='tight',dpi=200); plt.close(fig); display(_Img(filename=str(FIGD/name)))
def rows3(d):
    R=d['rows']; return (np.array([r['real'] for r in R]),np.array([r['shuffle'] for r in R]),np.array([r['fam'] for r in R]))
import numpy as np
z=None
for nm in ['biopwm_recog_indist_pwms.npz','biopwm_pwms_indist.npz']:
    p=OUT/nm
    if p.exists(): z=np.load(p,allow_pickle=True); break
if z is not None:
    P=z['pwm']; syms=list(z['syms'])
    if P.ndim==4: P=P[:,np.argmax(P.reshape(P.shape[0],P.shape[1],-1).var(2),axis=1)[0]] if False else P.mean(1)
    ic=(2+(np.clip(P,1e-9,1)*np.log2(np.clip(P,1e-9,1))).sum(-1)).sum(-1)
    pick=list(np.argsort(-ic)[:6])
    fig=ps.logo_grid([P[i] for i in pick],[syms[i] for i in pick],ncol=3)
    show(fig,'nb16_logos.png','Generated PWMs (read directly) - the interpretable latent')
else: display(Markdown('_PWMs pending_'))

## Conclusion

BioPWM makes the motif the explicit latent, so it is (1) **leakage-free** (protein->PWM->raw RNA bypasses the ~47% PARNET leak), (2) **interpretable by construction** (read the PWM; the only interpretability that survives the zero-shot shift where attention collapses), and (3) **RBP-specific** (beats the within-family shuffle in-distribution, where the cross-attn head was borderline). It is NOT a better raw predictor than the PARNET-conditioned head, and its protein deviation is best read on fine affinity assays (RBNS/RNAcompete), not coarse eCLIP. **Task-specific by inductive bias:** BioPWM is strong on **binary binding** (sharp motif occupancy) but **null on the nt-resolution profile** (section 5) - a PWM predicts motif location, not eCLIP coverage shape; per-residue cross-attn owns the profile. The principled multimodal fusion (PARNET bias + protein-BioPWM target, additive mixture) is the right architecture and keeps the gap leakage-free, but its win is on binary/affinity, not profile. This is the leakage-controlled, interpretable path that differentiates from CORAL. Next: the cross-method bake-off (notebook 17).

Claude-assisted; BioPWM per Ch206 (Stormo log-odds + Foat occupancy); recognition-code generator per the lab combined_biopwm.